1. Use requests and BeautifulSoup in Python to scrape the latest 10 headlines, authors, publication dates, and summaries from the technology section of a popular news site like NDTV Gadgets or The Verge, and print the extracted data.

In [ ]:
import requests
from bs4 import BeautifulSoup

def scrape_ndtv_tech():
    url = "https://gadgets360.com/news"   # NDTV Gadgets Technology news section
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    articles = soup.select("div.story")[:10]  # select first 10 articles

    for i, article in enumerate(articles, start=1):
        headline = article.find("h2").get_text(strip=True) if article.find("h2") else "No headline"
        author = article.find("span", class_="byline").get_text(strip=True) if article.find("span", class_="byline") else "Unknown"
        date = article.find("span", class_="dateline").get_text(strip=True) if article.find("span", class_="dateline") else "No date"
        summary = article.find("p").get_text(strip=True) if article.find("p") else "No summary"

        print(f"{i}. Headline: {headline}")
        print(f"   Author: {author}")
        print(f"   Date: {date}")
        print(f"   Summary: {summary}\n")

scrape_ndtv_tech()


2. Store the scraped news data (title, author, publication date, summary) into a pandas DataFrame and save it as a CSV file named tech_news.csv.

In [ ]:
import pandas as pd

# Suppose you already scraped your data into lists
titles = [
    "Headline 1", "Headline 2", "Headline 3", "Headline 4", "Headline 5",
    "Headline 6", "Headline 7", "Headline 8", "Headline 9", "Headline 10"
]
authors = [
    "Author A", "Author B", "Author C", "Author D", "Author E",
    "Author F", "Author G", "Author H", "Author I", "Author J"
]
dates = [
    "2026-10-13", "2026-10-12", "2026-10-11", "2026-10-10", "2026-10-09",
    "2026-10-08", "2026-10-07", "2026-10-06", "2026-10-05", "2026-10-04"
]
summaries = [
    "Summary of article 1", "Summary of article 2", "Summary of article 3",
    "Summary of article 4", "Summary of article 5", "Summary of article 6",
    "Summary of article 7", "Summary of article 8", "Summary of article 9",
    "Summary of article 10"
]

# Step 1: Create a DataFrame
df = pd.DataFrame({
    "Title": titles,
    "Author": authors,
    "Publication Date": dates,
    "Summary": summaries
})

# Step 2: Save to CSV
df.to_csv("tech_news.csv", index=False)

print("Saved scraped tech news to tech_news.csv")


3. Load your tech_news.csv file into pandas and perform basic EDA: display the number of articles, count how many articles each author wrote, and show the earliest and latest publication dates.

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("tech_news.csv")

# 1. Display the number of articles
num_articles = len(df)
print(f"Number of articles: {num_articles}")

# 2. Count how many articles each author wrote
author_counts = df["Author"].value_counts()
print("\nArticles per author:")
print(author_counts)

# 3. Show the earliest and latest publication dates
# Convert the 'Publication Date' column to datetime if not already
df["Publication Date"] = pd.to_datetime(df["Publication Date"], errors="coerce")

earliest_date = df["Publication Date"].min()
latest_date = df["Publication Date"].max()

print(f"\nEarliest publication date: {earliest_date}")
print(f"Latest publication date: {latest_date}")


4. Add a feature to your scraper so it skips any article where the summary is missing or empty before saving to CSV.<br><br><em><strong>Hint:</strong> Use an if condition to check for empty summaries during extraction.</em>

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_ndtv_tech():
    url = "https://gadgets360.com/news"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    articles = soup.select("div.story")[:20]  # grab more, but we'll filter

    scraped_data = []

    for article in articles:
        headline = article.find("h2").get_text(strip=True) if article.find("h2") else "No headline"
        author = article.find("span", class_="byline").get_text(strip=True) if article.find("span", class_="byline") else "Unknown"
        date = article.find("span", class_="dateline").get_text(strip=True) if article.find("span", class_="dateline") else "No date"
        summary = article.find("p").get_text(strip=True) if article.find("p") else ""

        # ✅ Skip if summary is missing or empty
        if summary.strip():
            scraped_data.append({
                "Title": headline,
                "Author": author,
                "Publication Date": date,
                "Summary": summary
            })

    # Convert to DataFrame
    df = pd.DataFrame(scraped_data)

    # Save to CSV
    df.to_csv("tech_news.csv", index=False)
    print("Saved tech news with non-empty summaries to tech_news.csv")

scrape_ndtv_tech()


5. Use ChatGPT to help you write a function that cleans the article summaries by removing extra whitespace and any HTML tags, then apply this function to your DataFrame before saving the CSV.<br><br><em><strong>Hint:</strong> Paste your current summary extraction code into ChatGPT and ask for suggestions on cleaning text data.</em>

In [ ]:
import pandas as pd
import re
from bs4 import BeautifulSoup

# Example cleaning function
def clean_summary(text):
    if pd.isna(text):
        return ""
    # Remove HTML tags using BeautifulSoup
    text = BeautifulSoup(text, "html.parser").get_text()
    # Collapse multiple spaces/newlines into single space
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Load your scraped CSV
df = pd.read_csv("tech_news.csv")

# Apply cleaning function to the Summary column
df["Summary"] = df["Summary"].apply(clean_summary)

# Save cleaned data back to CSV
df.to_csv("tech_news.csv", index=False)

print("Cleaned summaries and saved to tech_news.csv")
